In [1]:
# Model training for credit card fraud detection
import os
import json
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import argparse
from azureml.core import Dataset, Workspace, Datastore
from azureml.data.datapath import DataPath
import mlflow
import mlflow.sklearn

parser = argparse.ArgumentParser()
parser.add_argument("--input_data", type=str, default="~/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/data")
parser.add_argument("--output_model", type=str, default="~/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs")

args, _ = parser.parse_known_args()

INPUT_DIR = args.input_data
OUTPUT_DIR = args.output_model

print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)

# Get Azure ML workspace
ws = Workspace.from_config()

# Configure MLflow
mlflow.set_tracking_uri(ws.get_mlflow_tracking_uri())
mlflow.set_experiment("CreditCard_Fraud_Detection")

# Define training parameters
test_size = 0.2
random_state = 42
n_estimators = 100
max_depth = 10

print("\n" + "="*50)
print("MLflow Experiment: Credit Card Fraud Detection")
print("="*50)

# Start MLflow run
with mlflow.start_run(run_name="RandomForest_Training") as run:
    
    # Log parameters
    mlflow.log_param("test_size", test_size)
    mlflow.log_param("random_state", random_state)
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    
    print(f"\n✅ MLflow Run ID: {run.info.run_id}")
    
    # -----------------------------
    # Load data
    # -----------------------------
    print("\n📥 Loading data from Azure ML Datastore...")
    
    # Reference the datastore and path
    datastore = Datastore.get(ws, 'workspaceblobstore')
    datastore_path = 'UI/2026-04-04_090211_UTC/creditcard_train.csv'

    # Use DataPath to reference the file in the datastore
    dataset = Dataset.File.from_files(path=DataPath(datastore, datastore_path))
    downloaded_paths = dataset.download(target_path=INPUT_DIR, overwrite=True)
    data_path = downloaded_paths[0]

    data = pd.read_csv(data_path)
    
    # Log dataset info
    mlflow.log_param("dataset_shape", str(data.shape))
    print(f"✅ Dataset loaded: {data.shape[0]} rows, {data.shape[1]} columns")

    X = data.drop("Class", axis=1)
    y = data["Class"]
    
    # Log class distribution
    fraud_rate = (y == 1).sum() / len(y)
    mlflow.log_metric("fraud_rate", fraud_rate)
    print(f"✅ Fraud rate: {fraud_rate:.4f}")

    x_train, x_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    print(f"✅ Data split: {len(x_train)} train, {len(x_test)} test samples")

    # -----------------------------
    # Train model
    # -----------------------------
    print("\n🤖 Training model...")
    
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1
    )
    model.fit(x_train, y_train)
    
    print("✅ Model training completed")

    # -----------------------------
    # Evaluate
    # -----------------------------
    print("\n📊 Evaluating model...")
    
    y_pred = model.predict(x_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred)
    }

    # Log metrics to MLflow
    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(metric_name, metric_value)
        print(f"   {metric_name}: {metric_value:.4f}")

    # Log model to MLflow
    mlflow.sklearn.log_model(model, "model", registered_model_name=None)
    print("\n✅ Model logged to MLflow")

    # Log model artifact
    mlflow.log_artifact(os.path.join(OUTPUT_DIR, "metrics.json")) if os.path.exists(os.path.join(OUTPUT_DIR, "metrics.json")) else None

    # -----------------------------
    # Save model artifacts
    # -----------------------------
    print("\n💾 Saving model artifacts...")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    joblib.dump(model, os.path.join(OUTPUT_DIR, "model.pkl"))
    joblib.dump(x_test, os.path.join(OUTPUT_DIR, "x_test.pkl"))
    joblib.dump(y_test, os.path.join(OUTPUT_DIR, "y_test.pkl"))

    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=4)

    print("✅ Model and metrics saved locally")

    # Log artifact
    mlflow.log_artifact(os.path.join(OUTPUT_DIR, "metrics.json"))
    print("✅ Metrics artifact logged to MLflow")

    # Add tags
    mlflow.set_tag("dataset", "CreditCard")
    mlflow.set_tag("task", "fraud_detection")
    mlflow.set_tag("environment", "Azure_ML")

    # Print MLflow link
    print(f"\n🔗 View experiment at: {ws.get_mlflow_tracking_uri()}")
    print(f"   Run ID: {run.info.run_id}")

    # -----------------------------
    # Upload outputs to blob storage
    # -----------------------------
    print("\n☁️ Uploading outputs to blob storage...")
    
    datastore.upload(src_dir=OUTPUT_DIR, target_path='outputs', overwrite=True, show_progress=False)

    print("✅ Outputs uploaded to blob storage")
    
print("\n" + "="*50)
print("✅ Model training and logging completed successfully!")
print("="*50)

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
2026/04/08 12:30:22 INFO mlflow.tracking.fluent: Experiment with name 'CreditCard_Fraud_Detection' does not exist. Creating a new experiment.
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting 

Input: ~/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/data
Output: ~/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs

MLflow Experiment: Credit Card Fraud Detection

✅ MLflow Run ID: 5f1623ad-5f2e-4618-a208-e569e4ab2b9d

📥 Loading data from Azure ML Datastore...
{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
✅ Dataset loaded: 150000 rows, 32 columns
✅ Fraud rate: 0.0020
✅ Data split: 120000 train, 30000 test samples

🤖 Training model...
✅ Model training completed

📊 Evaluating model...
   accuracy: 0.9994
   precision: 1.0000
   f1_score: 0.8547
🏃 View run RandomForest_Training at: https://eastus2.api.azureml.ms/mlflow/v2.0/subscriptions/216d62c8-0f0c-4e5c-9cda-cc553e7ab186/resourceGroups/az03-aais-azu-mlops-ri-rg/providers/Microsoft.MachineLearningServices/workspaces/azuremlops/#/experiments/48bb97a8-a13b-4336-991e-28e103605786/runs/5f1623ad-5f2e-4618-a208-e569e4ab2b9d
🧪 View experi

MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: ''